For Audio

In [ ]:
SYSTEM_PROMPT = """You are an acoustic prosody analyst.

Hard rules:
- Use ONLY the provided features (numeric values and bins). Do not invent numbers or fields.
- If a field is not provided, you MUST NOT mention it.
- Do NOT mention any emotion labels or emotion words (anger, joy, sadness, fear, surprise, disgust, neutral).
- Output STRICT JSON only (no markdown, no extra text).

Output format (STRICT):
{
  "claims": [{"claim": "..."}, ...],
  "style_tags": ["..."],
  "one_liner": "...",
  "evidence_pool": { "k": v, ... }
}

Constraints:
- If pitch numeric features are provided (f0_mean_hz and optionally f0_std_hz/f0_slope_hz_per_sec), output exactly 3 claims.
- If pitch numeric features are NOT provided, output exactly 2 claims and do not mention pitch, pitch trend, or pitch variability.
- Each claim must be <= 16 words, natural language, no numbers inside claims.
- evidence_pool must contain ONLY the provided numeric features and bins that you used (no extra keys).
- Round numeric values to at most 3 decimals in evidence_pool.
- When using low/mid/high terms, align with the provided *_bin fields.
- style_tags: 3–6 snake_case tags, consistent with bins (e.g., low_loudness, high_voicing, short_duration).
- one_liner: <= 18 words, must be complete and end with a period.

Content planning rules:
If 3-claim case (pitch numeric provided):
1) Claim 1: duration/rate (may mention pauses only if pause_ratio is extreme).
2) Claim 2: loudness/dynamics (rms_* and bins).
3) Claim 3: pitch (must mention pitch using bins/trend only IF numeric pitch exists).

If 2-claim case (no pitch numeric):
- Do NOT mention pitch_trend or pitch_variability_bin.
- Use duration/rate + loudness/voicing as the two claims.

Pause rule:
- Mention pauses only if pause_ratio >= 0.23 or pause_ratio <= 0.19. Otherwise avoid pause wording.

one_liner rule:
- Must mention at least 2 of: duration_bin, loudness_bin, voicing_bin, rate_bin, noise_risk.
- Avoid generic openings like "The analysis reveals" or "distinct acoustic features".
"""

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}


For text and visual modal.

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import csv
import json
import base64
import requests

"""Your LLM API"""
API_KEY = "..."

"""This is the blacklist for the vision modality."""
EMO_BAN = re.compile(
    r"\b("
    r"angry|anger|furious|mad|irritated?|annoyed?|"
    r"sad|unhappy|depress(ed|ing)|upset|"
    r"fear|afraid|scare(d|y)|terrified|anxious|anxiety|"
    r"disgust(ed|ing)|gross|repuls(ed|ive)|"
    r"surprise(d)?|shocked|astonish(ed|ing)|amaze(d|ment)|"
    r"happy|happiness|glad|joy(ful)?|delight(ed|ful)|pleased|cheerful|"
    r"neutral|calm|relax(ed)?|content|"
    r"positive|negative"
    r")\b", re.IGNORECASE
)

def sanitize_caption(text: str, max_words: int = 40) -> str:
    if not text:
        return ""
    text = EMO_BAN.sub("[emotion-removed]", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    if len(words) > max_words:
        text = " ".join(words[:max_words]) + " ..."
    return text

def encode_image_to_data_url(path: str) -> str:
    ext = (os.path.splitext(path)[1] or "").lower()
    mime = "image/png"
    if ext in [".jpg", ".jpeg"]:
        mime = "image/jpeg"
    elif ext == ".webp":
        mime = "image/webp"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def describe_image(image_path: str) -> str:
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }
    data_url = encode_image_to_data_url(image_path)
    payload = {
        "model": "gpt-4o-mini",
        "temperature": 0.2,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a vision captioning assistant. "
                    "Write a concise, objective ENGLISH description of the image. "
                    "STRICTLY avoid any emotion/sentiment words (e.g., happy, sad, angry, surprised, calm, anxious). "
                    "Do not infer feelings or attitudes. Focus on number of people, positions, gaze, gestures, "
                    "objects, interactions, and scene context."
                ),
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Describe this image in one sentence, objectively and without emotion/sentiment words.",
                    },
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            },
        ],
    }
    r = requests.post(API_URL, headers=headers, json=payload, timeout=120)
    r.raise_for_status()
    js = r.json()
    raw = js["choices"][0]["message"]["content"].strip()
    return sanitize_caption(raw)

FUNCTION_MAP = {
    "trigger": "trigger",
    "outburst": "outburst",
    "protest": "protest",
    "clarify": "clarify",
    "explanation": "explanation",
    "avoidance": "avoidance",
    "cool_down": "cool_down",
    "surface_recovery": "cool_down",
    "humorous_relief": "humorous_relief",
    "decisive_action": "decisive_action",
    "intervention": "intervention",
}

DEFAULT_CLASS = "other"

def normalize_function(raw: str) -> str:
    if not raw:
        return DEFAULT_CLASS
    key = raw.strip().lower()
    return FUNCTION_MAP.get(key, DEFAULT_CLASS)

def extract_json_from_text(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {
        "semantic_explanation": text,
        "raw_function": "other",
        "turning": 0,
    }

def call_gpt_for_utterance(utterance: str, speaker: str = "") -> dict:
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }
    prompt = f"""
You are a dialog behavior analysis assistant.

Given one utterance from a conversation, output a JSON object with these fields:

- "semantic_explanation": A short ENGLISH explanation describing the FUNCTION of this utterance in the dialog 
  (e.g., triggering a sensitive topic, expressing disagreement, softening tension, avoiding deeper discussion,
   attempting clarification, confirming information, proposing a decision, etc.).
  IMPORTANT: Describe pragmatic function, NOT just restating the text or labeling emotion words.

- "raw_function": A short ENGLISH label (1–3 words, lowercase, snake_case) such as:
    - trigger
    - outburst
    - protest
    - clarify
    - explanation
    - avoidance
    - cool_down
    - humorous_relief
    - decisive_action
    - intervention
    - surface_recovery
    (or any other reasonable function)

- "turning": 0 or 1, indicating whether this utterance causes an important shift in the dialog flow
  (emotion, topic, or relationship).

Return ONLY a pure JSON object. Do NOT add anything else.

Utterance (speaker: {speaker}):
\"{utterance}\"
"""
    payload = {
        "model": "gpt-4o-mini",
        "temperature": 0.2,
        "messages": [
            {"role": "user", "content": prompt}
        ],
    }
    r = requests.post(API_URL, headers=headers, json=payload, timeout=120)
    r.raise_for_status()
    js = r.json()
    content = js["choices"][0]["message"]["content"]
    data = extract_json_from_text(content)
    data.setdefault("semantic_explanation", "")
    data.setdefault("raw_function", "other")
    data.setdefault("turning", 0)
    try:
        data["turning"] = int(data["turning"])
    except Exception:
        data["turning"] = 0
    return data

def pick_one_image_path(train_root: str, dialogue_id: int, utt_id: int):
    dir_name = f"dia{dialogue_id}_utt{utt_id}"
    dir_path = os.path.join(train_root, dir_name)
    if not os.path.isdir(dir_path):
        return None
    files = sorted(os.listdir(dir_path))
    full_imgs = [f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp")) and not f.endswith("_faces.jpg")]
    face_imgs = [f for f in files if f.endswith("_faces.jpg")]
    candidates = full_imgs if full_imgs else face_imgs
    if not candidates:
        return None
    mid_idx = len(candidates) // 2
    chosen = candidates[mid_idx]
    return os.path.join(dir_path, chosen)

def annotate_single_csv(input_csv: str, output_csv: str, train_root: str):
    with open(input_csv, "r", encoding="utf-8-sig") as f_in:
        reader = csv.DictReader(f_in)
        rows = list(reader)
        fieldnames = reader.fieldnames + [
            "semantic_explanation",
            "raw_function",
            "function",
            "turning",
            "vision_caption",
            "vision_image_path",
        ]
    with open(output_csv, "w", encoding="utf-8-sig", newline="") as f_out:
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()
        for i, row in enumerate(rows):
            utt = row.get("Utterance", "")
            speaker = row.get("Speaker", "")
            dia_str = row.get("Dialogue_ID", "0")
            utt_str = row.get("Utterance_ID", "0")
            try:
                dialogue_id = int(dia_str)
            except Exception:
                dialogue_id = 0
            try:
                utt_id = int(utt_str)
            except Exception:
                utt_id = 0
            if utt.strip():
                data = call_gpt_for_utterance(utt, speaker)
                sem = data["semantic_explanation"]
                raw_func = data["raw_function"]
                func = normalize_function(raw_func)
                turning = data["turning"]
            else:
                sem = ""
                raw_func = "other"
                func = "other"
                turning = 0
            img_path = pick_one_image_path(train_root, dialogue_id, utt_id)
            if img_path:
                try:
                    vision_cap = describe_image(img_path)
                except Exception:
                    vision_cap = ""
            else:
                vision_cap = ""
            row["semantic_explanation"] = sem
            row["raw_function"] = raw_func
            row["function"] = func
            row["turning"] = turning
            row["vision_caption"] = vision_cap
            row["vision_image_path"] = img_path or ""
            writer.writerow(row)
    print(f"Done: {output_csv}")

def annotate_all_dialogues(csv_dir: str, output_dir: str, train_root: str):
    os.makedirs(output_dir, exist_ok=True)
    for fname in sorted(os.listdir(csv_dir)):
        if not fname.startswith("dialogue_") or not fname.endswith(".csv"):
            continue
        in_path = os.path.join(csv_dir, fname)
        out_path = os.path.join(output_dir, fname)
        if os.path.exists(out_path):
            continue
        annotate_single_csv(in_path, out_path, train_root)

if __name__ == "__main__":
    CSV_DIR = "/root/autodl-tmp/train_Dia_csv"
    OUTPUT_DIR = "/root/autodl-tmp/Train_Dia_labeled"
    TRAIN_ROOT = "/root/autodl-tmp/MELD.Raw/train_splits"
    annotate_all_dialogues(CSV_DIR, OUTPUT_DIR, TRAIN_ROOT)